In [ ]:
from pathlib import Path
import numpy as np
import tifffile
from shutil import copy2

# ==============================
# paths and simple settings
# ==============================
DEM_DIR   = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10")
MOSAICS = {
    "R":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B04.tif",
    "G":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B03.tif",
    "B":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B02.tif",
    "NIR":  r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B08.tif",
    "NOISE":r"D:\Sunita_Thesis\Datasets\ProjectedNoise\StrassenLaerm_Tag_LV95_sigma1_32632.tif",
}
OUT_ROOT  = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_1km_DEMCropped")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# if you want to start from a specific tile, put its DEM filename stem here
# e.g. "swissalti3d_2025_2622-1250_2_2056_5728_10m_32632"
START_FROM_TILE = "swissalti3d_2025_2622-1250_2_2056_5728_10m_32632"      # or set to the tile_id string
LIMIT = None                # None = all tiles, or an integer for testing

# names of files we expect for each tile (for resume logic)
EXPECTED_SUFFIXES = ["__DEM.tif", "__R.tif", "__G.tif", "__B.tif", "__NIR.tif", "__NOISE.tif"]


# ==============================
# helper functions (same as before)
# ==============================
def _read_scale_tie(tf: tifffile.TiffFile):
    tags = tf.pages[0].tags
    scale = tags.get(33550)   # ModelPixelScaleTag
    tie   = tags.get(33922)   # ModelTiepointTag
    if not scale or not tie:
        raise RuntimeError("Missing GeoTIFF ModelPixelScale (33550) or Tiepoint (33922).")
    sx, sy = float(scale.value[0]), float(scale.value[1])
    i0, j0, _, X0, Y0, _ = [float(v) for v in tie.value[:6]]
    return (abs(sx), abs(sy)), (i0, j0), (X0, Y0)

def _origin_px_dims_nodata(tif_path):
    with tifffile.TiffFile(tif_path) as tf:
        (px_x, px_y), (i0, j0), (X0, Y0) = _read_scale_tie(tf)
        w = tf.pages[0].imagewidth
        h = tf.pages[0].imagelength
        nd_tag = tf.pages[0].tags.get(42113)  # GDAL_NODATA
        nd = None
        if nd_tag:
            val = nd_tag.value
            nd = float(val) if isinstance(val, (bytes, str)) else float(val)
    ox = X0 - i0 * px_x
    oy = Y0 + j0 * px_y
    return (ox, oy), (px_x, px_y), (w, h), nd

def _dem_ul_dims_nodata(dem_path):
    with tifffile.TiffFile(dem_path) as tf:
        (px_x, px_y), (i0, j0), (X0, Y0) = _read_scale_tie(tf)
        w = tf.pages[0].imagewidth
        h = tf.pages[0].imagelength
        nd_tag = tf.pages[0].tags.get(42113)
        nd = None
        if nd_tag:
            val = nd_tag.value
            nd = float(val) if isinstance(val, (bytes, str)) else float(val)
    ulx = X0 + i0 * px_x
    uly = Y0 - j0 * px_y
    return (ulx, uly), (px_x, px_y), (w, h), nd

def _window_from_world(ul_world, size_m, mosaic_origin, px_m):
    ulx, uly = ul_world
    ox, oy   = mosaic_origin
    px_x, px_y = px_m
    col_off = int(round((ulx - ox) / px_x))
    row_off = int(round((oy  - uly) / px_y))  # north-up
    width  = int(round(size_m[0] / px_x))
    height = int(round(size_m[1] / px_y))
    return row_off, col_off, height, width

def _write_geotiff_uncompressed(out_path, array, px, ul_world, geokeys_src_path, nodata=None):
    px_x, px_y = px
    ulx, uly = ul_world
    # copy CRS keys from source
    with tifffile.TiffFile(geokeys_src_path) as src:
        tags = src.pages[0].tags
        geokey = np.array(tags.get(34735).value, dtype=np.uint16) if tags.get(34735) else None
        geodbl = np.array(tags.get(34736).value, dtype=np.float64) if tags.get(34736) else None
        geoasc = tags.get(34737).value
        if isinstance(geoasc, bytes):
            geoasc = geoasc.decode('ascii', 'ignore')

    extratags = [
        (33550, 'd', 3, (float(px_x), float(px_y), 0.0), False),                 # ModelPixelScale
        (33922, 'd', 6, (0.0, 0.0, 0.0, float(ulx), float(uly), 0.0), False),    # ModelTiepoint
    ]
    if geokey is not None:
        extratags.append((34735, 'H', geokey.size, geokey, False))
    if geodbl is not None:
        extratags.append((34736, 'd', geodbl.size, geodbl, False))
    if geoasc is not None:
        extratags.append((34737, 's', len(geoasc), geoasc, False))
    if nodata is not None:
        nd_str = str(nodata)
        extratags.append((42113, 's', len(nd_str), nd_str, False))  # GDAL_NODATA

    tifffile.imwrite(
        out_path,
        array,
        photometric='minisblack',
        metadata=None,
        compression=None,
        extratags=extratags
    )

def _read_crop_with_padding(src_path, row_off, col_off, height, width, nodata=None):
    """Read a (height, width) window; pad outside with nodata."""
    with tifffile.TiffFile(src_path) as tf:
        src = tf.pages[0].asarray()
    mh, mw = src.shape[:2]

    if nodata is None:
        if np.issubdtype(src.dtype, np.floating):
            nodata = np.nan
        elif np.issubdtype(src.dtype, np.signedinteger):
            nodata = np.iinfo(src.dtype).min
        else:
            nodata = 0

    patch = np.full((height, width), nodata, dtype=src.dtype)

    # intersect with source image
    rs0 = max(0, row_off)
    cs0 = max(0, col_off)
    rs1 = min(mh, row_off + height)
    cs1 = min(mw, col_off + width)

    if rs1 > rs0 and cs1 > cs0:
        dr0 = rs0 - row_off
        dc0 = cs0 - col_off
        patch[dr0:dr0 + (rs1 - rs0), dc0:dc0 + (cs1 - cs0)] = src[rs0:rs1, cs0:cs1]

    return patch, nodata


# ==============================
# main processing
# ==============================

# read mosaic metadata once
mosaic_meta = {}
for name, p in MOSAICS.items():
    origin, px, dims, nd = _origin_px_dims_nodata(p)
    mosaic_meta[name] = {"origin": origin, "px": px, "dims": dims, "path": p, "nodata": nd}

# list DEM tiles
dem_files = sorted(DEM_DIR.glob("*.tif"))
dem_stems = [p.stem for p in dem_files]
total_tiles = len(dem_files)

# optional: start from a specific tile
start_index = 0
if START_FROM_TILE is not None:
    if START_FROM_TILE in dem_stems:
        start_index = dem_stems.index(START_FROM_TILE)
        dem_files = dem_files[start_index:]
        dem_stems = dem_stems[start_index:]
    else:
        print(f"WARNING: START_FROM_TILE '{START_FROM_TILE}' not found. Starting from the first tile.")

# optional limit for testing
if LIMIT is not None:
    dem_files = dem_files[:LIMIT]
    dem_stems = dem_stems[:LIMIT]

print(f"Found {total_tiles} DEM tiles in total.")
print(f"Will process {len(dem_files)} tiles starting from index {start_index}.")

processed_tiles = 0
skipped_tiles = 0

for i, dem_path in enumerate(dem_files, start=start_index + 1):
    tile_id = dem_path.stem
    tile_folder = OUT_ROOT / tile_id
    tile_folder.mkdir(parents=True, exist_ok=True)

    # check which outputs already exist (resume logic)
    expected_files = [tile_folder / f"{tile_id}{sfx}" for sfx in EXPECTED_SUFFIXES]
    if all(f.exists() for f in expected_files):
        print(f"[{i}/{total_tiles}] Skipping {tile_id} (all bands already present)")
        skipped_tiles += 1
        continue

    print(f"[{i}/{total_tiles}] Processing {tile_id} ...")

    # 1) DEM: copy if not already copied
    dem_out = tile_folder / f"{tile_id}__DEM.tif"
    if not dem_out.exists():
        copy2(dem_path, dem_out)

    # 2) read DEM georef (extent in world coords)
    ul_world, dem_px, (dem_w, dem_h), dem_nd = _dem_ul_dims_nodata(dem_path)
    dem_size_m = (dem_w * dem_px[0], dem_h * dem_px[1])

    # 3) crop each mosaic using DEM extent; only create missing outputs
    for name, meta in mosaic_meta.items():
        out_path = tile_folder / f"{tile_id}__{name}.tif"
        if out_path.exists():
            continue  # this band already done for this tile

        row_off, col_off, height, width = _window_from_world(
            ul_world=ul_world,
            size_m=dem_size_m,
            mosaic_origin=meta["origin"],
            px_m=meta["px"]
        )
        patch, nd = _read_crop_with_padding(
            meta["path"],
            row_off,
            col_off,
            height,
            width,
            nodata=meta["nodata"]
        )
        _write_geotiff_uncompressed(
            out_path,
            patch,
            meta["px"],
            ul_world,
            geokeys_src_path=meta["path"],
            nodata=nd
        )

    processed_tiles += 1
    print(f"    ✓ finished {tile_id}")

print(f"Done.")
print(f"Processed tiles: {processed_tiles}")
print(f"Skipped tiles:   {skipped_tiles}")
